# ch01 文本预处理（练习版）

> 按提示补全 `____`，跑通所有 assert。真值全部来自实跑（pandas 3.x，六条主线）。
>
> `VOCAB` / `STOPWORDS` / `SEG_RE` / `HAN_RE` / `MAX_WORD_LEN` 已给好；
> 挖空块内每条语句保持**单行**，控制流（`while` / `for` / `if`）都在块外。

In [ ]:
import re
from collections import Counter
from pathlib import Path

import pandas as pd

DATA = Path("data")
CSV = DATA / "synth_text.csv"

In [ ]:
# 文本数据一定要关掉默认的 NA 解析：否则 "" 和 "N/A" 会被静默吃掉（见 §1.2）
df = pd.read_csv(CSV, keep_default_na=False)

print("样本数", len(df), "| 列数", df.shape[1])
print("列名：", list(df.columns))
print("标签分布：", df["缺陷类型"].value_counts().to_dict())
print()
print(df[["工单编号", "设备类型", "缺陷类型", "缺陷描述"]].head(4).to_string(index=False))

## 任务 1：读取口径（`keep_default_na`）

同一份 CSV 读两遍，看缺失值个数差多少，并想清楚**哪个数字才是你想要的**。

In [ ]:
# 同一份 CSV 读两遍：一遍用默认设置，一遍关掉 NA 解析
raw = pd.read_csv(CSV)
safe = pd.read_csv(CSV, keep_default_na=False)

n_raw = int(raw["缺陷描述"].isna().sum())
n_safe = int(safe["缺陷描述"].isna().sum())

print("默认读取的缺失值：", n_raw)
print("关闭 NA 解析的缺失值：", n_safe)
print("默认读取的 dtype：", raw["缺陷描述"].dtype)

# ---- 验收 ----
assert n_raw == 12, f"默认读取应吃掉 12 条，实际 {n_raw}"
assert n_safe == 0, f"关掉 NA 解析后不应有缺失，实际 {n_safe}"

## 任务 2：脏数据体检

先数清楚，再动手洗。这两行是后面所有清洗的验收基准。

In [ ]:
desc = df["缺陷描述"]

n_blank = int((desc.str.strip() == "").sum())

n_html = int(desc.str.contains("<p>", regex=False).sum())

n_url = int(desc.str.contains("http", regex=False).sum())
n_half = int(desc.str.contains(",", regex=False).sum())
n_bang = int(desc.str.contains("！！", regex=False).sum())
n_ws = int(((desc.str.contains("  ", regex=False)) | (desc != desc.str.strip())).sum())
n_bad = int(desc.str.strip().isin(["", "N/A", "-", "无"]).sum())

print(f"空/纯空白     {n_blank}")
print(f"含 HTML      {n_html}")
print(f"含 URL       {n_url}")
print(f"含半角逗号     {n_half}")
print(f"含重复标点     {n_bang}")
print(f"含多余/首尾空白  {n_ws}")
print(f"无效占位符     {n_bad}")
print(f"去重后         {desc.nunique()}")

# ---- 验收 ----
assert n_blank == 12, f"空/纯空白应为 12，实际 {n_blank}"
assert n_html == 107, f"含 HTML 应为 107，实际 {n_html}"
assert n_url == 82 and n_half == 131 and n_bang == 41
assert n_ws == 180 and n_bad == 30 and desc.nunique() == 575

## 任务 3：正则清洗五步管道

按注释分三段补全：去 HTML/URL、标点归一、去汉字间空格。
第 3 段的零宽断言写法是关键 —— 它必须**不碰** `STATION_E_02` 里的下划线。

In [ ]:
def clean(text: str) -> str:
    """把一条工单描述洗成干净的中文句子。"""
    s = re.sub(r"<[^>]+>", "", text)
    s = re.sub(r"https?://\S+", "", s)

    s = s.replace(",", "，")
    s = re.sub(r"([，。！？])\1+", r"\1", s)

    s = re.sub(r"(?<=[\u4e00-\u9fff])\s+(?=[\u4e00-\u9fff])", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

In [ ]:
desc_clean = df["缺陷描述"].map(clean)

print("清洗后平均字符数：", round(float(desc_clean.str.len().mean()), 6))
print("清洗前平均字符数：", round(float(df["缺陷描述"].str.len().mean()), 6))
print()
print("样例对照：")
i = 0
print("  原文：", repr(df["缺陷描述"][i]))
print("  清洗：", repr(desc_clean[i]))

# ---- 验收 ----
assert int((desc_clean.str.strip() == "").sum()) == 12
assert int(desc_clean.str.contains("<", regex=False).sum()) == 0
assert int(desc_clean.str.contains("http", regex=False).sum()) == 0
assert int(desc_clean.str.contains(",", regex=False).sum()) == 0
assert abs(float(desc_clean.str.len().mean()) - 28.513333) < 1e-5

## 任务 4：贪婪 vs 非贪婪

两个正则都能「去标签」，但一个会把正文也吃掉。

In [ ]:
greedy_demo = "<p>文本</p><p>更多</p>"

greedy = re.sub(r"<.*>", "", greedy_demo)
lazy = re.sub(r"<.*?>", "", greedy_demo)

print("贪婪  <.*>  ->", repr(greedy))
print("非贪婪 <.*?> ->", repr(lazy))

# ---- 验收 ----
assert greedy == "", "贪婪会连中间的正文字符一起吃掉"
assert lazy == "文本更多", "非贪婪才只去标签、留下正文"

## 任务 5：正向最大匹配

内层 `while` 已经在帮你缩短候选长度了，你只需要**收词 + 移指针**。
注意指针要右移「匹配长度」，不是 1。

In [ ]:
VOCAB = [
    # 设备
    "变压器", "断路器", "隔离开关", "避雷器", "电流互感器", "绝缘子", "套管", "母线",
    # 缺陷信号词（五类互不重叠）
    "渗油", "漏油", "油迹", "油污", "滴油", "油位下降",
    "锈蚀", "生锈", "铁锈", "漆面剥落", "镀锌层脱落",
    "破损", "裂纹", "碎裂", "缺口", "掉瓷", "箱体变形",
    "发热", "温度偏高", "过热", "烫手", "温升异常", "过热保护",
    "异物", "鸟巢", "塑料袋", "树枝", "风筝线搭挂",
    # 动作与描述
    "发现", "巡检", "位于", "存在", "缺陷", "一般", "严重", "危急",
    "接报", "详见", "保护", "动作",
    # 高频短语（故意把整句当词，看最大匹配怎么处理长词）
    "请安排抢修", "请尽快核实", "已通知运维班", "请列入本周计划",
    "需停电处理", "请核实后反馈",
    # 教学用词：专门用来演示「正向 ≠ 逆向」
    "研究", "研究生", "生命", "命", "起源",
]

In [ ]:
# 脚手架：词表、停用词、正则常量（两版都有，你只填 @@todo 块）
VOCAB_SET = set(VOCAB)
MAX_WORD_LEN = max(len(w) for w in VOCAB)

# 粗切分：连续 ASCII 字母数字下划线 / 连续汉字 / 单个其他字符
SEG_RE = re.compile(r"[A-Za-z0-9_]+|[\u4e00-\u9fff]+|[^\sA-Za-z0-9_\u4e00-\u9fff]")
# 注意这里的 + ：没有 + 就只能匹配单个汉字，而 fullmatch 要求整串匹配（见 §1.8）
HAN_RE = re.compile(r"[\u4e00-\u9fff]+")

STOPWORDS = {
    "的", "了", "在", "是", "和", "与", "及", "或", "也", "就", "都",
    "，", "。", "！", "？", "、", "（", "）", "【", "】", "：", "；", " ",
}

print("词表", len(VOCAB), "个词 | 最长", MAX_WORD_LEN, "字 | 停用词", len(STOPWORDS), "个")

In [ ]:
def fmm(seg: str) -> list[str]:
    """正向最大匹配：从左往右扫，每次取词表里能匹配上的最长词。"""
    out, i = [], 0
    while i < len(seg):
        length = min(MAX_WORD_LEN, len(seg) - i)
        while length > 1 and seg[i:i + length] not in VOCAB_SET:
            length -= 1
        out.append(seg[i:i + length])
        i += length
    return out

In [ ]:
print("变压器渗油      ->", "/".join(fmm("变压器渗油")))
print("巡检发现渗油     ->", "/".join(fmm("巡检发现渗油")))
print("研究生命起源     ->", "/".join(fmm("研究生命起源")))

# ---- 验收 ----
assert fmm("变压器渗油") == ["变压器", "渗油"]
assert fmm("巡检发现渗油") == ["巡检", "发现", "渗油"]

## 任务 6：逆向最大匹配 + 方向对比

从右往左扫，注意收词是 `insert(0, ...)`（保持从左到右的输出顺序）。

最后一个 assert 会告诉你一个反直觉的事实：**真实工单里两个方向不分歧**。

In [ ]:
def bmm(seg: str) -> list[str]:
    """逆向最大匹配：从右往左扫，每次取词表里能匹配上的最长词。"""
    out, i = [], len(seg)
    while i > 0:
        length = min(MAX_WORD_LEN, i)
        while length > 1 and seg[i - length:i] not in VOCAB_SET:
            length -= 1
        out.insert(0, seg[i - length:i])
        i -= length
    return out

In [ ]:
fmm_ans = fmm("研究生命起源")
bmm_ans = bmm("研究生命起源")

print("正向：", "/".join(fmm_ans))
print("逆向：", "/".join(bmm_ans))
print("词数：", len(fmm_ans), "vs", len(bmm_ans))

both_diff = sum(1 for s in desc_clean if fmm(s) != bmm(s))
print("600 条工单里两方向不一致的条数：", both_diff)

# ---- 验收 ----
assert bmm("研究生命起源") == ["研究", "生命", "起源"]
assert fmm("研究生命起源") != bmm("研究生命起源")
assert both_diff == 0, "真实工单里两种方向其实并不分歧"

## 任务 7：中文夹英文编号的两级切分

判断「是不是纯汉字段」要用 `fullmatch`，不是 `match`。

In [ ]:
def tokenize(text: str) -> list[str]:
    """先粗切分，汉字段走最大匹配，其余（英文 / 数字 / 标点）整体入列。"""
    out = []
    for seg in SEG_RE.findall(text):
        pieces = fmm(seg) if HAN_RE.fullmatch(seg) else [seg]
        out.extend(pieces)
    return out

In [ ]:
sample = desc_clean[0]
print("样例：", sample)
print("分词：", tokenize(sample))
print()
print("如果跳过粗切分、直接对整句做最大匹配：")
print(" ", fmm(sample)[:12], "...")

# ---- 验收 ----
assert tokenize(sample) == [
    "STATION_E_02", "绝缘子", "渗油", "，", "请核实后反馈", "，", "请安排抢修",
]
assert len(fmm(sample)) > len(tokenize(sample)), "裸 FMM 会把编号逐字符切碎"

## 任务 8：停用词过滤（别忘了标点）

In [ ]:
tokens = tokenize(desc_clean[0])

kept = [w for w in tokens if w not in STOPWORDS and w.strip()]

print("分词结果：", tokens)
print("去停用词：", kept)
print("token 数：", len(tokens), "->", len(kept))

# ---- 验收 ----
assert len(tokens) == 7 and len(kept) == 5
assert "，" not in kept

In [ ]:
all_tokens = [
    w
    for s in desc_clean
    for w in tokenize(s)
    if w not in STOPWORDS and w.strip()
]
counter = Counter(all_tokens)

print("全语料 token 总数：", len(all_tokens))
print("词表大小（去停用词后）：", len(counter))
print("Top 10：", counter.most_common(10))

# ---- 验收 ----
assert len(all_tokens) == 3461
assert len(counter) == 75

## 任务 9：n-gram

In [ ]:
def ngrams(seq, n):
    """返回所有长度为 n 的连续子序列（元组形式）。"""
    return [tuple(seq[i:i + n]) for i in range(len(seq) - n + 1)]

In [ ]:
base = kept
print("序列：", base)
print("1-gram：", ngrams(base, 1))
print("2-gram：", ngrams(base, 2))
print("3-gram：", ngrams(base, 3))
print()
print("字级 2-gram「电力设备」：", ngrams(list("电力设备"), 2))

# ---- 验收 ----
assert len(ngrams(base, 1)) == 5
assert len(ngrams(base, 2)) == 4
assert len(ngrams(base, 3)) == 3
assert ngrams(list("电力设备"), 2) == [("电", "力"), ("力", "设"), ("设", "备")]

## 任务 10：简化版词干提取

命中后缀就切掉，但**要留够 3 个字符** —— 否则 `"is"` 这种短词会被砍成空串。

In [ ]:
SUFFIXES = ("ational", "ization", "iveness", "fulness", "ousness", "ing", "ed", "ly", "es", "s")


def stem(word: str) -> str:
    """简化版 Porter 词干：命中后缀且剩余长度 >= 3 就切掉。"""
    w = word.lower()
    hit = next((s for s in SUFFIXES if w.endswith(s) and len(w) - len(s) >= 3), None)
    return w[: -len(hit)] if hit else w

In [ ]:
words = ["running", "studies", "organization", "cars", "boxes", "quickly", "flying"]
for w in words:
    print(f"  {w:14s} -> {stem(w)}")

print()
print("注意：running -> runn（不是 run）。词干只负责砍后缀，不保证结果是真词；")
print("      running -> run 属于词形还原（lemmatization），要靠词表或词性标注。")

# ---- 验收 ----
assert stem("running") == "runn", "简化词干砍掉 ing 后得到 runn，不会自动补成 run"
assert stem("organization") == "organ"
assert stem("cars") == "car"
assert stem("quickly") == "quick"
assert stem("flying") == "fly"

## 自查清单

- [ ] 文本列读取关掉了默认 NA 解析
- [ ] 清洗后 `<` / `http` / 半角逗号 / `！！` 四种残留全部为 0
- [ ] 平均字符数从 34.986667 降到 28.513333
- [ ] 正向最大匹配：指针右移的是**匹配长度**
- [ ] 逆向最大匹配：收词用 `insert(0, ...)`
- [ ] 粗切分用 `fullmatch` 判断纯汉字段
- [ ] 停用词表里**包含中文标点**
- [ ] n-gram 的起点范围是 `len(seq) - n + 1`
- [ ] 词干只剩 3 个字符时不再继续砍
- [ ] 能说出「研究生命起源」为什么两个方向结果不同